# Importing a public dataset : template

**Dr. Panagiotis Sakagiannis, Dr. Alexandros Marantis**

## About this notebook

A blank version of the larvaworld public-dataset import notebook, ready to point at a new
dataset. Everything that changes from one dataset to the next lives in **Section 1** and the
first cells of **Section 2**; the rest runs unchanged.

Replace the description below with the dataset you are importing :

**The dataset.** *TODO - where the recordings come from, what was tracked, how long each
recording is, and whether there was any stimulus.*

**The biological question.** *TODO - what the original study asked, and whether the dataset
holds one larva group or several distinct groups or conditions to compare. If it holds
several, list them in a small table like the one below.*

| group | condition | expectation |
|---|---|---|
| **control** | *TODO* | *TODO* |
| **treatment** | *TODO* | *TODO* |

A worked example of this notebook is
`import_feeding_state_locomotion_dataset.ipynb`.

## Setup

Importing larvaworld initializes its configuration registry : some components are loaded
from disc and the rest are built on the fly. `VERBOSE = 1` makes the import report what it
is doing, which is worth watching the first time.

In [ ]:
%matplotlib inline

from pathlib import Path

from IPython.display import display

import larvaworld
from larvaworld.lib import reg, sim, util
from larvaworld.lib.reg.generators import ReplayConf

larvaworld.VERBOSE = 1

# The name of this experiment. It labels the imported datasets and the output folders.
EXPERIMENT_NAME = "MyExperiment"

MEDIA_DIR = Path(f"./media/{EXPERIMENT_NAME}")
plot_dir = (MEDIA_DIR / "plots").as_posix()
video_dir = (MEDIA_DIR / "videos").as_posix()

# Rendering the replay videos needs ffmpeg and takes several minutes.
MAKE_VIDEOS = False

ds = []  # the imported datasets, filled in further below

## Section 1 : Get the data

Download and extract the dataset wherever you like - the import reads the tracker's files
where they are, so nothing needs to be moved or renamed.

*TODO - record the source here : the repository link, the file to download, and any
unpacking step the reader has to perform.*

- **Record** : *TODO*
- **File to download** : *TODO*

Fill in the four values below. `RAW_FOLDER` and `EXPERIMENT` split the path so that
`RAW_FOLDER/EXPERIMENT/<group>` is the folder holding one group's recordings :

```text
<RAW_FOLDER>/
└── <EXPERIMENT>/
    ├── <group 1>/     <- one subfolder per recording session
    ├── <group 2>/
    └── ...
```

If your dataset has no group level, put every recording under a single group folder and
list that one name.

In [ ]:
# TODO where you extracted the download
DOWNLOAD_ROOT = Path.home() / "Downloads" / "MyDataset"

# TODO the folder holding the experiment, and the experiment folder inside it
RAW_FOLDER = (DOWNLOAD_ROOT / "path" / "to").as_posix()
EXPERIMENT = "experiment_folder"

# TODO the groups to compare and the color each one gets in every plot
palette = {"control": "black", "treatment": "red"}
gIDs = list(palette)


# The recording folders of a group : subfolders that actually hold data. Hidden and
# metadata folders that archives and network drives leave behind (".DS_Store",
# "@eaDir", ...) are skipped, as are empty folders.
def recording_folders(group_dir):
    return sorted(
        p.name
        for p in Path(group_dir).iterdir()
        if p.is_dir() and not p.name.startswith((".", "@")) and any(p.rglob("*"))
    )


available = [g for g in gIDs if (Path(RAW_FOLDER) / EXPERIMENT / g).is_dir()]
if len(available) != len(gIDs):
    print(f"Data not found under :\n  {RAW_FOLDER}/{EXPERIMENT}")
    print(f"missing groups : {[g for g in gIDs if g not in available]}")
else:
    for g in gIDs:
        print(
            f"{g:12s} : {len(recording_folders(Path(RAW_FOLDER) / EXPERIMENT / g))} recordings"
        )
DATA_AVAILABLE = len(available) == len(gIDs)

## Section 2 : Import to larvaworld

Three things have to be specified for the import to work : **where the data is**, **which
tracker wrote it**, and **which tracks to keep**. Everything else has a sensible default.

### What the tracker recorded, and what it did not

Before importing, it is worth knowing which properties of a recording are written down
somewhere and which are not. For this dataset - and, in our experience, for most published
tracking data - the picture is this :

| property | stated in the archive? | larvaworld can derive it |
|---|---|---|
| recording duration | yes, in the tracker's settings file | not needed |
| stimulus protocol | yes, in the tracker's settings file | not needed |
| **frame rate** | **no** | **yes**, from the timestamps |
| **number of midline points** | **no** | **yes**, from the coordinates |
| **arena dimensions** | **no** | partly, see below |
| **pixel-to-millimetre scale** | **no** | no - you have to know it |

The highlighted rows are the ones that matter for the import, and none of them is stated
anywhere. larvaworld therefore derives what it can from the data itself :

- **Frame rate.** Many trackers record at a variable rate, so a single nominal frame rate
  does not describe them. When that is the case the timestep is measured from the
  timestamps of the recording and used for the whole import, including the stored dataset.
  Pass `estimate_dt=False` to keep the nominal value instead.
- **Midline points.** Counted from the data and used whenever it disagrees with the
  expected number. Pass `estimate_midline_points=False` to switch this off.
- **Arena dimensions.** Estimated from the area the animals actually covered, which makes
  it a *lower bound* : larvae that never reach the rim make the arena look smaller than it
  is. It is therefore **off by default**, and worth enabling only when the real arena is
  unknown. Pass `estimate_arena_dimensions=True`.

**The spatial scale is the one thing you must bring yourself.** If a tracker exports pixels
rather than millimetres, no amount of inspection of the coordinates will reveal the
conversion factor. A quick sanity check settles which case you are in : the summed length
of a larva's midline should be a few millimetres for a third-instar larva.

### Which tracker wrote it

Pick the lab format matching the tracker your data came from. There is a short section on
lab formats at the end of the notebook.

In [ ]:
# TODO the lab format matching your tracker
LAB_FORMAT = "Jovanic"

lf = reg.conf.LabFormat.get(LAB_FORMAT)

### Which tracks to keep, and what to compute

The values below are a reasonable starting point for a dispersal analysis. Adjust
`time_slice` to the window you care about and `min_duration_in_sec` to how much of a track
you consider usable.

In [ ]:
constraints = util.AttrDict(
    {
        "match_ids": False,  # use the tracker's own identities
        "interpolate_ticks": True,  # resample onto a regular time grid
        "min_duration_in_sec": 20,  # TODO discard tracks shorter than this
        "time_slice": (0, 60),  # TODO the window to analyse, in seconds
    }
)

enr_kws = util.AttrDict(
    {
        "proc_keys": ["angular", "spatial"],
        "anot_keys": ["bout_detection"],
        "traj2origin": True,
        "tor_durs": [20],
        "dsp_starts": [0],
        "dsp_stops": [40, 60],  # TODO match these to your time_slice
    }
)

### Putting it together

The three `estimate_*` arguments are the answer to the metadata question above. The
defaults are usually right; set them explicitly so that a reader of your notebook can see
what was measured and what was assumed.

In [ ]:
refIDs = [f"{EXPERIMENT_NAME}.{g}" for g in gIDs]

kws = {
    # Where the data is.
    "raw_folder": RAW_FOLDER,
    "parent_dir": EXPERIMENT,
    "source_ids": gIDs,
    # How the imported datasets are identified and stored.
    "group_id": EXPERIMENT_NAME,
    "refIDs": refIDs,
    "colors": [palette[g] for g in gIDs],
    "save_dataset": True,
    # What to measure from the data rather than assume.
    "estimate_dt": True,  # ignored if the tracker has a constant framerate
    "estimate_midline_points": True,
    "estimate_arena_dimensions": False,  # set True only if the real arena is unknown
    "enrich_conf": enr_kws,
    **constraints,
}

In [ ]:
if DATA_AVAILABLE:
    ds = lf.import_datasets(**kws)

    for d in ds:
        print(
            f"{d.id:12s} : {d.config.N} larvae, dt={d.config.dt:.4f} s, {d.config.Npoints} midline points"
        )

Check the log before moving on : the number of larvae per group, the timestep and the
midline point count are the three things most likely to reveal a mis-specified import.

### Reloading in a later session

In later sessions skip everything above and start from the cell below.

In [ ]:
if not ds:
    if all(refID in reg.conf.Ref.confIDs for refID in refIDs):
        ds = [reg.loadRef(id=refID, load=True) for refID in refIDs]
        print("Loaded :", [d.id for d in ds])
    else:
        print("These datasets have not been imported yet. Run Section 2 first.")

## Section 3 : Data analysis and plotting

larvaworld ships a library of plotting routines, each registered under a short name. You
pick one by name and hand it the datasets you want compared - the group colors and labels
are taken from the datasets themselves, so every figure is consistent.

In [ ]:
# The available plots, by their unique IDs
print(reg.graphs.ks)

In [ ]:
# Arguments shared by every plot below. Figures are also written to `plot_dir`.
plot_kws = {"datasets": ds, "save_to": plot_dir, "show": False, "subfolder": None}

### The trajectories

First, simply what the larvae did : their paths inside the dish over the analysed window.

In [ ]:
if ds:
    display(reg.graphs.run("trajectories", **plot_kws))

The same trajectories, but each one translated so that it starts at the origin, and colored
by group. This removes the arbitrary starting position of each animal and makes the *shape
and extent* of the paths directly comparable between groups.

In [ ]:
if ds:
    display(
        reg.graphs.run("trajectories", mode="origin", single_color=True, **plot_kws)
    )

### Endpoint metrics

A boxplot of endpoint metrics - one value per larva, summarising its whole track. Each plot
routine has a sensible default selection, but you can always name the metrics you want by
their short keys :

| key | metric |
|---|---|
| `l` | body length |
| `fsv` | crawling frequency |
| `sv_mu` | mean scaled crawling speed |
| `str_sd_mu` | mean scaled distance covered per stride |
| `run_tr`, `pau_tr` | fraction of time spent running / pausing |
| `tor20_mu` | mean tortuosity over 20 s windows |
| `dsp_0_40_fin` | dispersal reached after 40 s |
| `b_mu`, `bv_mu` | mean body bending and bending velocity |

In [ ]:
if ds:
    display(
        reg.graphs.run(
            "endpoint box",
            ks=[
                "l",
                "fsv",
                "sv_mu",
                "str_sd_mu",
                "run_tr",
                "pau_tr",
                "tor20_mu",
                "dsp_0_40_fin",
                "b_mu",
                "bv_mu",
            ],
            **plot_kws,
        )
    )

And a composite figure summarising exploration behavior across the groups.

In [ ]:
if ds:
    display(reg.graphs.run("exploration summary", **plot_kws))

### Dispersal

Dispersal is the distance of a larva from where it started. We compare the groups on it
in three increasingly informative ways.

**1. As an endpoint statistic.** The mean, final and maximum dispersal reached during the
analysed window - one number per larva, summarised as a boxplot per group. This captures
the outcome but says nothing about how it was reached.

In [ ]:
if ds:
    display(
        reg.graphs.run(
            "endpoint box",
            ks=["dsp_0_60_mu", "dsp_0_60_fin", "dsp_0_60_max"],
            **plot_kws,
        )
    )

**2. As a timecourse.** The dispersal of the larvae from their starting point plotted
against time, showing both the mean and the variance of each group. This is the temporal
evolution we are after : it shows not just how far the groups got, but how fast, and how
consistently.

In [ ]:
if ds:
    # The default time range is 0-40 seconds.
    display(reg.graphs.run("dispersal", **plot_kws))

In [ ]:
if ds:
    # The same over the full analysed minute.
    display(reg.graphs.run("dispersal", range=(0, 60), **plot_kws))

**3. Alongside the paths that produced it.** The summary versions place the timecourse next
to the corresponding trajectories, which makes the link between curve and behavior
immediate.

In [ ]:
if ds:
    display(reg.graphs.run("dispersal summary", **plot_kws))

In [ ]:
if ds:
    display(reg.graphs.run("dispersal summary", range=(0, 60), **plot_kws))

## Section 4 : Visualize the dataset

A *replay* is a simulation whose agents are driven by recorded data instead of a model. It
gives you the same visualization tools you would use on a simulation - here, the
trajectories of all larvae of a group, transposed to a common origin and drawn as
accumulating trails.

Rendering needs `ffmpeg` (installed with larvaworld via `imageio_ffmpeg`) and takes a few
minutes per group, so it is off by default. Set `MAKE_VIDEOS = True` in the Setup cell to
run it.

In [ ]:
def run_replay(d):
    """Render one dataset's tracks to a video file in `video_dir`."""
    screen_kws = {
        "vis_mode": "video",
        "show_display": False,
        "draw_contour": False,
        "draw_midline": False,
        "draw_centroid": False,
        "visible_trails": True,
        "save_video": True,
        "fps": 1,
        "video_file": d.id,
        "media_dir": video_dir,
    }
    replay_conf = ReplayConf(
        transposition="origin", time_range=(0, 60), track_point=d.c.point_idx
    ).nestedConf
    rep = sim.ReplayRun(
        dataset=d,
        parameters=replay_conf,
        id=f"{d.id}_replay",
        screen_kws=screen_kws,
    )
    return rep.run()

In [ ]:
if MAKE_VIDEOS and ds:
    for d in ds:
        run_replay(d)

Finally the videos are stacked side by side into a single one, giving a direct visual
comparison of the groups.

In [ ]:
if MAKE_VIDEOS and ds:
    from larvaworld.lib.util.combining import combine_videos

    combine_videos(file_dir=video_dir, save_as="combined.mp4")
    print(f"Written to {video_dir}/combined.mp4")

## A few words on the lab format

Every tracker writes its own files, so larvaworld reads each one through a named **lab
format**. A lab format knows how a lab's files are laid out and how their contents must be
preprocessed, which is why the import above needed nothing more than a folder and a name.

| lab format | suits data that looks like |
|---|---|
| `Jovanic` | one file per recorded quantity, all animals stacked together |
| `Schleyer` | one file per animal, plus per-dish metadata |
| `Berni`, `Arguello` | one file per animal, columns in a fixed order |
| `DeepLabCut` | DeepLabCut CSV/HDF5 exports, one file per video |

Two things follow from this :

- **If one of them matches your tracker**, this notebook works on your own data with only
  the first cell changed.
- **If none does**, a new lab format can be described and registered, after which your data
  imports like any other.

A lab format carries nominal values for things like the frame rate and the arena, because
they are usually constant for a lab. They describe the lab, not any particular recording,
which is why the import prefers what it can measure in the data itself.

## References

*TODO - cite the dataset and the study it belongs to. Both, if they are separate
publications, and in the form the repository asks for.*